# 24 · Guardrails that fail loudly

## Goal

Wire DLP (connectors and MCP alike), agent quarantine, secret handling via
connection references, content moderation, and injection defence — and
give every one of them a test that fails when the guard is removed. A
guard nobody tests is a comment pretending to be a control.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.clients import get_copilot_client
settings = load_settings()
client = get_copilot_client(settings, delegated=True)


## Concept

Every guardrail below gets the same treatment: a positive test (the guard
blocks what it should) *and*, where feasible, a negative control (removing
the guard makes the test fail) — otherwise you don't actually know the
test is testing anything. `evals/golden_cases.json`'s `governance`-tagged
cases are written for exactly this; `gov-01` through `gov-03` each pair
with a specific control below.

Per finding #10, MCP servers ride connector infrastructure, so the same
DLP policy from `infra/terraform/platform/dlp.tf` governs both — `13`
already proved this once; this notebook makes it a standing, tested
guarantee rather than a one-off demonstration.

Per finding #5, external/preview model enablement is itself a governance
surface with four independent switches — `22`'s prereqs cell is the
control for that one; this notebook doesn't repeat it, just references it.


## Build


### Content moderation + secret handling — asserted in copilot.yaml, not left implicit


In [ ]:
import yaml
from pathlib import Path
spine = Path("../agents/contract-renewal-desk")
copilot_yaml = yaml.safe_load((spine / "copilot.yaml").read_text())
copilot_yaml["contentModeration"] = {"level": "strict"}
copilot_yaml["responseCaching"] = {"enabled": True, "ttlSeconds": 300}  # notebooks/03's citation cases still must pass with caching on
(spine / "copilot.yaml").write_text(yaml.dump(copilot_yaml, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(spine)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### `pac copilot quarantine` — the kill switch, exercised, not just documented


In [ ]:
from csx.pac import copilot_quarantine

copilot_quarantine("crd_contract-renewal-desk", settings.get("DATAVERSE_ENV_ID"), enable=True)
quarantined_reply = client.ask_question("What can you help me with?")
print("while quarantined:", quarantined_reply.text[:200])
assert "unavailable" in quarantined_reply.text.lower() or "quarantine" in quarantined_reply.text.lower(), "quarantine should visibly block responses"

copilot_quarantine("crd_contract-renewal-desk", settings.get("DATAVERSE_ENV_ID"), enable=False)
restored_reply = client.ask_question("What can you help me with?")
print("after lifting quarantine:", restored_reply.text[:200])


## Verify

Same harness, same golden set, every notebook.


Every guard, positive test — this is the block from `evals/golden_cases.json#governance`.


In [ ]:
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

gov_cases = load_golden(tags=["governance"])
gov_suite = run_suite(client, cases=gov_cases, credit_meter=meter, min_pass_rate=1.0)  # governance cases get a 100% bar, not 80%


Negative control — prove `gov-03` fails without the moderation guard, so we know the test isn't a no-op.


In [ ]:
import yaml
copilot_yaml = yaml.safe_load((spine / "copilot.yaml").read_text())
copilot_yaml["contentModeration"] = {"level": "off"}
(spine / "copilot.yaml").write_text(yaml.dump(copilot_yaml, sort_keys=False))
from csx.pac import copilot_push
import subprocess
copilot_push(spine)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)

control_case = next(c for c in gov_cases if c["id"] == "gov-03-content-safety")
control_reply = client.ask_question(control_case["prompt"])
guard_removed_still_declines = "can't" in control_reply.text.lower() or "not able" in control_reply.text.lower()
print(f"with moderation OFF, still declined? {guard_removed_still_declines}")
# If this is now True even with moderation off, the golden case isn't
# actually testing the moderation guard — tighten it before trusting it.

# restore
copilot_yaml["contentModeration"] = {"level": "strict"}
(spine / "copilot.yaml").write_text(yaml.dump(copilot_yaml, sort_keys=False))
copilot_push(spine)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Cost


In [ ]:
meter.report_cost("24", budget=settings.get("COPILOT_CREDIT_BUDGET"),
                   delta_credits=(gov_suite.total_credits + 2),
                   note="governance suite (100% bar) + quarantine drill + negative-control run")


## Teardown


In [ ]:
print("No teardown — content moderation, response caching, and the quarantine capability all persist into 25's production posture.")
